# FD-21: M2 v1 (microsoft/unixcoder-base @1024) — Kaggle 2xT4
Шаги workflow R10: пилот 100 шагов → overfit 200x20 → полный прогон 3 эпохи → усечение → предсказания → оценка. Выбор эпохи — по PR-AUC **val_real**. Подробнее — `docs/kaggle_fd21_guide.md`.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/o1uh/FreshDoc.git
%cd /kaggle/working/FreshDoc
!git checkout feat/FD-21-m2-train
!pip install -q -r requirements.txt
import os
os.environ['NCCL_P2P_DISABLE'] = '1'
!nvidia-smi

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
for m in ['microsoft/unixcoder-base']:
    tok = AutoTokenizer.from_pretrained(m)
    mod = AutoModel.from_pretrained(m).to('cuda')
    out = mod(**tok('def test(): pass', return_tensors='pt').to('cuda'))
    print(m, tuple(out.last_hidden_state.shape))
print('smoke OK')

In [ ]:
!python scripts/train_m2.py --config config/train.yaml --data_path /kaggle/input/freshdoc-dataset/dataset_v1.parquet --output_dir /kaggle/working/checkpoints/m2_pilot --pilot_steps 100

In [ ]:
!python scripts/train_m2.py --config config/train.yaml --data_path /kaggle/input/freshdoc-dataset/dataset_v1.parquet --output_dir /kaggle/working/checkpoints/m2_overfit --overfit_test

In [ ]:
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml --data_path /kaggle/input/freshdoc-dataset/dataset_v1.parquet --output_dir /kaggle/working/checkpoints/m2_run1

In [ ]:
!python scripts/check_truncation.py --model microsoft/unixcoder-base --max_len 1024 --dataset /kaggle/input/freshdoc-dataset/dataset_v1.parquet --out /kaggle/working/m2_truncation.json
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/m2_run1/best --dataset /kaggle/input/freshdoc-dataset/dataset_v1.parquet --out /kaggle/working/m2.csv
!python scripts/evaluate.py --dataset /kaggle/input/freshdoc-dataset/dataset_v1.parquet --pred M0=reports/preds/m0.csv --pred M1=reports/preds/m1.csv --pred M2=/kaggle/working/m2.csv --out_metrics /kaggle/working/metrics_m2.json --out_table /kaggle/working/e1_m2.csv